# Pythia8 + FastJet + ROOT (Jupyter)

Generates pp → QCD dijet events with Pythia8, clusters jets with FastJet anti-$k_t$,
and writes per-jet variables into a ROOT TTree via ROOT C++.

**Load order matters in Jupyter** (no `module load` autoload here):  
`hepyy.load('root')` must run **before** fastjet/pythia8 so ROOT's bundled
cppyy becomes `sys.modules['cppyy']`. All packages then share ROOT's single cling
instance — `import ROOT` works cleanly alongside pythia8/fastjet.

Prerequisites: `heyy install root fastjet pythia8`

In [ ]:
# ROOT must load first — its lib/ is inserted at sys.path[0] so ROOT's
# cppyy.py wins the next `import cppyy`. fastjet/pythia8 then share ROOT's cling.
#
# If ROOT is not managed by hepyy (system/conda install), replace
# hepyy.load('root') with a bare `import ROOT` instead.
import hepyy
hepyy.load('root')
hepyy.load('fastjet')
hepyy.load('pythia8')

In [ ]:
import array
import cppyy
import ROOT
import pythia8
import fastjet
import numpy as np
import matplotlib.pyplot as plt

# Sanity check: cppyy should come from ROOT's installation, not pip-cppyy.
print(f"cppyy from: {cppyy.__file__}")

PseudoJetVec = cppyy.gbl.std.vector[fastjet.PseudoJet]

## ROOT output: TFile + TTree

In [ ]:
outfile = ROOT.TFile("pythia_jets_nb.root", "RECREATE")
tree    = ROOT.TTree("jets", "anti-kt R=0.4 jets from Pythia8 pp dijets")

b_event      = array.array('i', [0])
b_pt         = array.array('f', [0.])
b_eta        = array.array('f', [0.])
b_phi        = array.array('f', [0.])
b_e          = array.array('f', [0.])
b_m          = array.array('f', [0.])
b_nconst     = array.array('i', [0])
b_is_leading = array.array('i', [0])

tree.Branch("event",      b_event,      "event/I")
tree.Branch("pt",         b_pt,         "pt/F")
tree.Branch("eta",        b_eta,        "eta/F")
tree.Branch("phi",        b_phi,        "phi/F")
tree.Branch("e",          b_e,          "e/F")
tree.Branch("m",          b_m,          "m/F")
tree.Branch("nconst",     b_nconst,     "nconst/I")
tree.Branch("is_leading", b_is_leading, "is_leading/I")

## Configure Pythia8

In [ ]:
pythia = pythia8.Pythia()
pythia.readString("Beams:eCM = 13000.")
pythia.readString("HardQCD:all = on")
pythia.readString("PhaseSpace:pTHatMin = 20.")
pythia.readString("Next:numberShowEvent = 0")
pythia.readString("Print:quiet = on")
pythia.init()

## Event loop

In [ ]:
R        = 0.4
pt_min   = 20.0
n_events = 500
jet_def  = fastjet.JetDefinition(fastjet.antikt_algorithm, R)

# Python lists for matplotlib — filled in parallel with the TTree
all_pt         = []
all_eta        = []
leading_pt     = []
n_jets_per_evt = []

print(f"Generating {n_events} events, anti-kt R={R}, pt > {pt_min} GeV")

for i_event in range(n_events):
    if not pythia.next():
        continue

    particles = PseudoJetVec()
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        if p.isFinal() and p.isVisible():
            particles.push_back(fastjet.PseudoJet(p.px(), p.py(), p.pz(), p.e()))

    if particles.size() == 0:
        continue

    cs   = fastjet.ClusterSequence(particles, jet_def)
    jets = fastjet.sorted_by_pt(cs.inclusive_jets(pt_min))
    n_jets_per_evt.append(len(jets))

    for j, jet in enumerate(jets):
        b_event[0]      = i_event
        b_pt[0]         = jet.pt()
        b_eta[0]        = jet.eta()
        b_phi[0]        = jet.phi()
        b_e[0]          = jet.e()
        b_m[0]          = jet.m()
        b_nconst[0]     = len(cs.constituents(jet))
        b_is_leading[0] = 1 if j == 0 else 0
        tree.Fill()

        all_pt.append(jet.pt())
        all_eta.append(jet.eta())
        if j == 0:
            leading_pt.append(jet.pt())

    if i_event < 3 and jets:
        print(f"  event {i_event}: {int(particles.size())} particles, {len(jets)} jets, "
              f"leading pt={jets[0].pt():.1f} GeV")

pythia.stat()

## Write ROOT file

In [ ]:
n_entries = tree.GetEntries()
outfile.Write()
outfile.Close()

print(f"Wrote pythia_jets_nb.root  ({n_events} events, {n_entries} jet entries)")
print("Branches: event, pt, eta, phi, e, m, nconst, is_leading")
print()
print("Read back with ROOT:")
print('  root -l pythia_jets_nb.root')
print('  jets->Draw("pt>>h(50,0,200)","is_leading==1")')
print()
print("Read back with uproot:")
print('  import uproot')
print('  with uproot.open("pythia_jets_nb.root") as f:')
print('      pt = f["jets"]["pt"].array()')

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f'Pythia8 pp → dijets @ 13 TeV,  anti-$k_t$  $R={R}$,  '
    f'$p_T > {pt_min}$ GeV  ({n_events} events)'
)

# Leading jet pT
ax = axes[0]
bins = np.linspace(20, 200, 45)
ax.hist(leading_pt, bins=bins, histtype='step', linewidth=1.5, color='steelblue')
ax.set_xlabel(r'Leading jet $p_T$ [GeV]')
ax.set_ylabel('Jets / bin')
ax.set_title(r'Leading jet $p_T$')
ax.set_yscale('log')

# All-jet eta
ax = axes[1]
ax.hist(all_eta, bins=50, range=(-5, 5), histtype='step', linewidth=1.5, color='seagreen')
ax.set_xlabel(r'Jet $\eta$')
ax.set_ylabel('Jets / bin')
ax.set_title(r'Inclusive jet $\eta$')

# Jet multiplicity
ax = axes[2]
max_n = max(n_jets_per_evt) if n_jets_per_evt else 5
ax.hist(n_jets_per_evt, bins=range(0, max_n + 2), align='left',
        histtype='step', linewidth=1.5, color='darkorange')
ax.set_xlabel('Jets per event')
ax.set_ylabel('Events')
ax.set_title('Jet multiplicity')
ax.set_xticks(range(0, max_n + 1))

plt.tight_layout()
plt.savefig('demo_pythia_fastjet_root.png', dpi=150)
plt.show()
print('Saved demo_pythia_fastjet_root.png')

## Read back from ROOT file

Demonstrates that the ROOT TTree written above can be read back via ROOT C++ in the same session,
or via [uproot](https://uproot.readthedocs.io) without ROOT at all.

In [ ]:
# Read back via ROOT C++ (same cling session — file was already closed above)
f2 = ROOT.TFile("pythia_jets_nb.root")
t2 = f2.Get("jets")
print(f"Entries in tree: {t2.GetEntries()}")
print(f"Branches: {[b.GetName() for b in t2.GetListOfBranches()]}")

ROOT.gROOT.SetBatch(True)
c = ROOT.TCanvas("c", "", 600, 400)
t2.Draw("pt>>hpt(50,20,200)", "is_leading==1", "goff")
hpt = ROOT.gDirectory.Get("hpt")
print(f"Leading-jet pT histogram: {int(hpt.GetEntries())} entries, "
      f"mean = {hpt.GetMean():.1f} GeV")
f2.Close()

In [ ]:
# Read back via uproot (pure Python, no ROOT C++ needed)
try:
    import uproot
    with uproot.open("pythia_jets_nb.root") as f:
        pt         = f["jets"]["pt"].array(library="np")
        is_leading = f["jets"]["is_leading"].array(library="np")
    print(f"uproot read {len(pt)} jet entries")
    print(f"Leading jets: {is_leading.sum()}  mean pt = {pt[is_leading==1].mean():.1f} GeV")
except ImportError:
    print("uproot not installed — skip (pip install uproot)")